In [ ]:
!pip install sentence-transformers faiss-cpu numpy -q
from sentence_transformers import SentenceTransformer, InputExample, losses
from torch.utils.data import DataLoader
import faiss
import numpy as np
import json

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
with open('/content/drive/MyDrive/LegalRAG/data/embedding_train_synthetic.json', 'r') as f:
    embedding_train = json.load(f)

print(f"Toplam soru: {len(embedding_train)}")
print(f"İlk örnek keyleri: {list(embedding_train[0].keys())}")
print(f"Örnek: {embedding_train[0]}")

Toplam soru: 4648
İlk örnek keyleri: ['soru', 'chunk_id', 'kanun', 'madde_no']
Örnek: {'soru': "Ceza Kanunu'nun temel amacı nedir ve hangi değerleri korumayı hedefler?", 'chunk_id': 'türk_ceza_kanunu__madde_1__0', 'kanun': 'Türk Ceza Kanunu', 'madde_no': 'MADDE 1'}


In [ ]:
print("Reranker eğitim verisi hazırlanıyor...")

sorular = [s['soru'] for s in embedding_train]
dogru_ids = [s['chunk_id'] for s in embedding_train]

# Soruları encode et
soru_emb = model_tur2.encode(sorular, batch_size=64, normalize_embeddings=True, show_progress_bar=True)
_, indices = index_tur2.search(soru_emb, 20)

reranker_train = []
for i, (soru, dogru_id) in enumerate(zip(sorular, dogru_ids)):
    dogru_idx = chunk_id_to_idx.get(dogru_id, -1)
    if dogru_idx == -1:
        continue

    top_20 = indices[i].tolist()

    # Pozitif
    if dogru_idx in top_20:
        reranker_train.append({
            "soru": soru,
            "chunk": chunk_texts[dogru_idx],
            "label": 1
        })

    # Negatif — top-20'den doğru chunk çıkar, kalan 5'i al
    negatifler = [idx for idx in top_20 if idx != dogru_idx][:5]
    for idx in negatifler:
        reranker_train.append({
            "soru": soru,
            "chunk": chunk_texts[idx],
            "label": 0
        })

pozitif = sum(1 for r in reranker_train if r['label'] == 1)
negatif = sum(1 for r in reranker_train if r['label'] == 0)
print(f"Toplam: {len(reranker_train)} örnek")
print(f"Pozitif: {pozitif}, Negatif: {negatif}")
print(f"Oran: 1 pozitif / {negatif//pozitif} negatif")

Reranker eğitim verisi hazırlanıyor...


Batches:   0%|          | 0/73 [00:00<?, ?it/s]

Toplam: 27888 örnek
Pozitif: 4648, Negatif: 23240
Oran: 1 pozitif / 5 negatif


In [ ]:
!pip install transformers accelerate -q

from torch.utils.data import Dataset
from transformers import AutoModelForSequenceClassification, AutoTokenizer, TrainingArguments, Trainer
import torch

class RerankerDataset(Dataset):
    def __init__(self, data, tokenizer, max_length=512):
        self.data = data
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        encoding = self.tokenizer(
            item['soru'],
            item['chunk'],
            truncation=True,
            max_length=self.max_length,
            padding='max_length',
            return_tensors='pt'
        )
        return {
            'input_ids': encoding['input_ids'].squeeze(),
            'attention_mask': encoding['attention_mask'].squeeze(),
            'labels': torch.tensor(item['label'], dtype=torch.float)
        }

# Model ve tokenizer yükle
print("BGE reranker yükleniyor...")
model_name = "BAAI/bge-reranker-v2-m3"
tokenizer = AutoTokenizer.from_pretrained(model_name)
reranker_model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=1)

print(f"GPU: {torch.cuda.memory_allocated()/1e9:.2f} GB")

# Dataset
import random
random.seed(42)
random.shuffle(reranker_train)

dataset = RerankerDataset(reranker_train, tokenizer)
print(f"Dataset hazır: {len(dataset)} örnek")

BGE reranker yükleniyor...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/795 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.27G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

GPU: 1.62 GB
Dataset hazır: 27888 örnek


In [ ]:
from transformers import TrainingArguments, Trainer
import torch.nn as nn

class RerankerTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits.squeeze(-1)
        loss = nn.BCEWithLogitsLoss()(logits, labels)
        return (loss, outputs) if return_outputs else loss

training_args = TrainingArguments(
    output_dir='/content/drive/MyDrive/LegalRAG/models/bge_reranker_ft',
    num_train_epochs=3,
    per_device_train_batch_size=16,
    warmup_steps=100,
    weight_decay=0.01,
    logging_steps=100,
    save_strategy='epoch',
    learning_rate=2e-5,
    fp16=True,
    dataloader_num_workers=2,
    report_to='none'
)

trainer = RerankerTrainer(
    model=reranker_model,
    args=training_args,
    train_dataset=dataset,
)

print("BGE Reranker FT başlıyor...")
print(f"GPU: {torch.cuda.memory_allocated()/1e9:.2f} GB")
trainer.train()

# Kaydet
reranker_model.save_pretrained('/content/drive/MyDrive/LegalRAG/models/bge_reranker_ft')
tokenizer.save_pretrained('/content/drive/MyDrive/LegalRAG/models/bge_reranker_ft')
print("✅ BGE Reranker FT tamamlandı")

BGE Reranker FT başlıyor...
GPU: 3.89 GB


Step,Training Loss
100,0.342735
200,0.299730
300,0.315512
400,0.303525
500,0.310404
600,0.278428
700,0.292443
800,0.278665
900,0.299691
1000,0.281821


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ BGE Reranker FT tamamlandı


In [ ]:
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer

# Fine-tuned reranker yükle
print("Fine-tuned BGE reranker yükleniyor...")
ft_reranker = AutoModelForSequenceClassification.from_pretrained(
    '/content/drive/MyDrive/LegalRAG/models/bge_reranker_ft'
)
ft_reranker_tokenizer = AutoTokenizer.from_pretrained(
    '/content/drive/MyDrive/LegalRAG/models/bge_reranker_ft'
)
ft_reranker.eval()
ft_reranker.to('cuda')
print(f"GPU: {torch.cuda.memory_allocated()/1e9:.2f} GB")

def rerank(soru, chunk_listesi, model, tokenizer, batch_size=32):
    skorlar = []
    for i in range(0, len(chunk_listesi), batch_size):
        batch = chunk_listesi[i:i+batch_size]
        encoding = tokenizer(
            [soru] * len(batch),
            batch,
            truncation=True,
            max_length=512,
            padding=True,
            return_tensors='pt'
        ).to('cuda')
        with torch.no_grad():
            outputs = model(**encoding)
            batch_skorlar = outputs.logits.squeeze(-1).cpu().tolist()
            if isinstance(batch_skorlar, float):
                batch_skorlar = [batch_skorlar]
            skorlar.extend(batch_skorlar)
    return skorlar

def metrikleri_olc_reranker(faiss_model, faiss_index, reranker_model, reranker_tokenizer, gold_161, chunk_id_to_idx, chunk_texts, top_k=20):
    sorular = [s['question'] for s in gold_161]

    dogru_chunk_idler = []
    for s in gold_161:
        ids = []
        for source in s['gold_sources']:
            ids.extend(source.get('matched_chunk_ids', []))
        dogru_chunk_idler.append(ids)

    soru_emb = faiss_model.encode(sorular, batch_size=64, normalize_embeddings=True, show_progress_bar=True)
    _, indices = faiss_index.search(soru_emb, top_k)

    recall_5 = recall_10 = mrr = ndcg = 0

    for i, dogru_ids in enumerate(dogru_chunk_idler):
        dogru_idxler = [chunk_id_to_idx[cid] for cid in dogru_ids if cid in chunk_id_to_idx]
        if not dogru_idxler:
            continue

        top_k_idxler = indices[i].tolist()
        top_k_chunks = [chunk_texts[idx] for idx in top_k_idxler]

        # Rerank
        skorlar = rerank(sorular[i], top_k_chunks, reranker_model, reranker_tokenizer)
        sirali = sorted(zip(top_k_idxler, skorlar), key=lambda x: x[1], reverse=True)
        sirali_idxler = [idx for idx, _ in sirali]

        if any(idx in sirali_idxler[:5] for idx in dogru_idxler):  recall_5 += 1
        if any(idx in sirali_idxler[:10] for idx in dogru_idxler): recall_10 += 1
        for rank_idx, idx in enumerate(sirali_idxler):
            if idx in dogru_idxler:
                rank = rank_idx + 1
                mrr += 1 / rank
                ndcg += 1 / math.log2(rank + 1)
                break

    n = len(gold_161)
    return {
        "Recall@5":  round(recall_5/n, 4),
        "Recall@10": round(recall_10/n, 4),
        "MRR":       round(mrr/n, 4),
        "nDCG@10":   round(ndcg/n, 4)
    }

# Test et
with open('/content/drive/MyDrive/LegalRAG/data/gold_test_normalized_matched_161.json', 'r') as f:
    gold_161 = json.load(f)

print("\n=== FT Dense + FT BGE Reranker ===")
sonuclar = metrikleri_olc_reranker(model_tur2, index_tur2, ft_reranker, ft_reranker_tokenizer, gold_161, chunk_id_to_idx, chunk_texts)
for m, v in sonuclar.items():
    print(f"{m}: {v}")

print("\n=== KARŞILAŞTIRMA ===")
print(f"{'Metrik':<12} {'FT Dense':>10} {'FT Dense+BGE':>13}")
ft_dense = {"Recall@5": 0.8696, "Recall@10": 0.8758, "MRR": 0.7997, "nDCG@10": 0.8191}
for m in ["Recall@5", "Recall@10", "MRR", "nDCG@10"]:
    print(f"{m:<12} {ft_dense[m]:>10.4f} {sonuclar[m]:>13.4f}")

Fine-tuned BGE reranker yükleniyor...


Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

GPU: 10.72 GB

=== FT Dense + FT BGE Reranker ===


Batches:   0%|          | 0/3 [00:00<?, ?it/s]

Recall@5: 0.882
Recall@10: 0.882
MRR: 0.8443
nDCG@10: 0.8552

=== KARŞILAŞTIRMA ===
Metrik         FT Dense  FT Dense+BGE
Recall@5         0.8696        0.8820
Recall@10        0.8758        0.8820
MRR              0.7997        0.8443
nDCG@10          0.8191        0.8552


In [ ]:
from tqdm import tqdm

print("=== SENTETİK GOLD TEST — FT Dense + FT BGE Reranker ===")

sorular = [s['soru'] for s in gold_synthetic]
dogru_chunk_idler = [s['chunk_id'] for s in gold_synthetic]

soru_emb = model_tur2.encode(sorular, batch_size=64, normalize_embeddings=True, show_progress_bar=True)
_, indices = index_tur2.search(soru_emb, 20)

recall_5 = recall_10 = mrr = ndcg = 0

for i, dogru_id in enumerate(tqdm(dogru_chunk_idler)):
    dogru_idx = chunk_id_to_idx.get(dogru_id, -1)
    if dogru_idx == -1:
        continue

    top_k_idxler = indices[i].tolist()
    top_k_chunks = [chunk_texts[idx] for idx in top_k_idxler]

    skorlar = rerank(sorular[i], top_k_chunks, ft_reranker, ft_reranker_tokenizer)
    sirali = sorted(zip(top_k_idxler, skorlar), key=lambda x: x[1], reverse=True)
    sirali_idxler = [idx for idx, _ in sirali]

    if dogru_idx in sirali_idxler[:5]:  recall_5 += 1
    if dogru_idx in sirali_idxler[:10]: recall_10 += 1
    if dogru_idx in sirali_idxler:
        rank = sirali_idxler.index(dogru_idx) + 1
        mrr += 1 / rank
        ndcg += 1 / math.log2(rank + 1)

n = len(gold_synthetic)
print(f"\nRecall@5:  {round(recall_5/n, 4)}")
print(f"Recall@10: {round(recall_10/n, 4)}")
print(f"MRR:       {round(mrr/n, 4)}")
print(f"nDCG@10:   {round(ndcg/n, 4)}")

=== SENTETİK GOLD TEST — FT Dense + FT BGE Reranker ===


Batches:   0%|          | 0/37 [00:00<?, ?it/s]

100%|██████████| 2324/2324 [10:28<00:00,  3.70it/s]


Recall@5:  0.9281
Recall@10: 0.9479
MRR:       0.8457
nDCG@10:   0.8724


In [ ]:
%%capture
!pip install -q "transformers>=4.48.0" trl==0.11.4 peft==0.13.2 "accelerate>=1.1.0" sentencepiece datasets
!pip install -q "bitsandbytes>=0.46.1" --upgrade
!pip install -q sentence-transformers faiss-cpu rouge-score anthropic

In [ ]:
import json, torch, faiss, numpy as np, math
from sentence_transformers import SentenceTransformer

with open('/content/drive/MyDrive/LegalRAG/data/mevzuat_chunked0v2_normalized.json', 'r') as f:
    chunks = json.load(f)

with open('/content/drive/MyDrive/LegalRAG/data/gold_test_normalized_matched_161.json', 'r') as f:
    gold_161 = json.load(f)

chunk_id_to_idx = {c['metadata']['chunk_id']: i for i, c in enumerate(chunks)}
chunk_texts = [c['text'] for c in chunks]

print(f"Chunk: {len(chunks)}, Gold test: {len(gold_161)}")

Chunk: 2324, Gold test: 161
